In [1]:
import numpy as np
import pandas as pd

from src.data import load_flattened_data
from src.features import get_train_class
from src.modeling import build_train_table, train_catboost, build_eval_table
from src.evaluate import evaluate_catboost, evaluate_metrics, eval_n_class

# Baseline

In [2]:
df = load_flattened_data("train")
df = get_train_class(df, return_df=True)
df.head()

,vehicle_id,time_step,171_0,666_0,427_0,837_0,167_0,167_1,167_2,167_3,...,Spec_2,Spec_3,Spec_4,Spec_5,Spec_6,Spec_7,length_of_study_time_step,in_study_repair,tte,class_label
0,0,11.2,167985.0,10787.0,7413813.0,2296.0,4110.0,1296420.0,1628265.0,630345.0,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,498.8,0
1,0,11.4,167985.0,10787.0,7413813.0,2296.0,4111.0,1302855.0,1628265.0,630345.0,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,498.6,0
2,0,19.6,331635.0,14525.0,13683604.0,2600.0,NaN,NaN,NaN,NaN,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,490.4,0
3,0,20.2,354975.0,15015.0,14540449.0,2616.0,NaN,NaN,NaN,NaN,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,489.8,0
4,0,21.0,365550.0,15295.0,14966985.0,2720.0,NaN,NaN,NaN,NaN,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,489.0,0


In [3]:
X, y, groups, weights = build_train_table(df)

In [4]:
df_val = load_flattened_data("validation")
X_val, y_val, groups_val = build_eval_table(df_val)

In [5]:
print("Always class 0 on validation set")
_ = eval_n_class(0, print_metrics=True)

Always class 0 on validation set
Predictive cost: 57400
Classification report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.99      4910
           1       0.00      0.00      0.00        16
           2       0.00      0.00      0.00        14
           3       0.00      0.00      0.00        30
           4       0.00      0.00      0.00        76

    accuracy                           0.97      5046
   macro avg       0.19      0.20      0.20      5046
weighted avg       0.95      0.97      0.96      5046

Confusion matrix:
 [[4910    0    0    0    0]
 [  16    0    0    0    0]
 [  14    0    0    0    0]
 [  30    0    0    0    0]
 [  76    0    0    0    0]]


In [6]:
print("Always class 4 on validation set")
_ = eval_n_class(4, print_metrics=True)

Always class 4 on validation set
Predictive cost: 49566
Classification report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00      4910
           1       0.00      0.00      0.00        16
           2       0.00      0.00      0.00        14
           3       0.00      0.00      0.00        30
           4       0.02      1.00      0.03        76

    accuracy                           0.02      5046
   macro avg       0.00      0.20      0.01      5046
weighted avg       0.00      0.02      0.00      5046

Confusion matrix:
 [[   0    0    0    0 4910]
 [   0    0    0    0   16]
 [   0    0    0    0   14]
 [   0    0    0    0   30]
 [   0    0    0    0   76]]


# Catboost

In [10]:
model = train_catboost(X, y, n_estimators=10, random_state=0)

Learning rate set to 0.5
0:	learn: 0.3724814	total: 868ms	remaining: 7.81s
1:	learn: 0.2594708	total: 1.81s	remaining: 7.25s
2:	learn: 0.2044189	total: 2.59s	remaining: 6.04s
3:	learn: 0.1759728	total: 3.42s	remaining: 5.13s
4:	learn: 0.1602582	total: 4.25s	remaining: 4.25s
5:	learn: 0.1518279	total: 5.02s	remaining: 3.35s
6:	learn: 0.1470398	total: 5.77s	remaining: 2.47s
7:	learn: 0.1438099	total: 6.55s	remaining: 1.64s
8:	learn: 0.1413933	total: 7.34s	remaining: 815ms
9:	learn: 0.1401909	total: 8.12s	remaining: 0us


In [11]:
vehicle_last_idx = groups.groupby(groups).tail(1).index
cost_tr, report_tr, confusion_tr = evaluate_catboost(model, X.loc[vehicle_last_idx], y[vehicle_last_idx], print_metrics=True)

Predictive cost: 1088010
Classification report:
               precision    recall  f1-score   support

           0       0.90      1.00      0.95     21287
           1       0.00      0.00      0.00        31
           2       0.00      0.00      0.00        70
           3       0.00      0.00      0.00       161
           4       0.86      0.00      0.01      2000

    accuracy                           0.90     23549
   macro avg       0.35      0.20      0.19     23549
weighted avg       0.89      0.90      0.86     23549

Confusion matrix:
 [[21286     0     0     0     1]
 [   31     0     0     0     0]
 [   70     0     0     0     0]
 [  161     0     0     0     0]
 [ 1991     1     1     1     6]]


In [12]:
cost_val, report_val, confusion_val = evaluate_catboost(model, X_val, y_val, print_metrics=True)

Predictive cost: 57420
Classification report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.99      4910
           1       0.00      0.00      0.00        16
           2       0.00      0.00      0.00        14
           3       0.00      0.00      0.00        30
           4       0.00      0.00      0.00        76

    accuracy                           0.97      5046
   macro avg       0.19      0.20      0.20      5046
weighted avg       0.95      0.97      0.96      5046

Confusion matrix:
 [[4908    0    0    0    2]
 [  16    0    0    0    0]
 [  14    0    0    0    0]
 [  30    0    0    0    0]
 [  76    0    0    0    0]]


# Survival analysis